# ViT-Small — Amazonian Geoglyph Detection
**Model 6 of 7 — Phase 1 Classical Benchmark**

Vision Transformer (ViT) — fundamentally different from all previous models.
No convolutional layers at all. Splits the 224x224 image into 196 patches
of 16x16 pixels, treats each patch as a token, and uses self-attention to
learn global relationships between all patches simultaneously.

Key differences from CNNs:
- No local feature detection — sees the whole image at once
- No translation equivariance — must learn spatial structure from data
- Output: 384-dim CLS token embedding (not spatial feature maps)
- No GlobalAveragePooling needed — CLS token is already flat
- Requires vit-keras library — not in keras.applications

Library: vit-keras (lightweight TF-native ViT implementation)

In [ ]:
!pip install transformers -q
print("transformers installed")

transformers installed


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

POS_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
NEG_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/negative"
VALID_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

def count_images(folder_path):
    files = [f for f in os.listdir(folder_path)
             if os.path.splitext(f)[1].lower() in VALID_EXT]
    return len(files)

n_pos = count_images(POS_DIR)
n_neg = count_images(NEG_DIR)
print(f"Positive: {n_pos} | Negative: {n_neg} | Total: {n_pos + n_neg}")
print("Drive mounted OK")

Mounted at /content/drive
Positive: 530 | Negative: 2181 | Total: 2711
Drive mounted OK


In [ ]:
import tensorflow_hub as hub
print("TF Hub version:", hub.__version__)
print("Ready — no extra installs needed")

TF Hub version: 0.16.1
Ready — no extra installs needed


In [ ]:
import os, random, numpy as np, pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ViT-S/16 pretrained on ImageNet-21k from TF Hub
VIT_URL = "https://tfhub.dev/sayakpaul/vit_s16_fe/1"

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))
print("ViT-S/16 URL ready:", VIT_URL)

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
ViT-S/16 URL ready: https://tfhub.dev/sayakpaul/vit_s16_fe/1


## Step 2 — Splits & Class Weights
Identical to all previous models. SEED=42, 70/15/15 stratified split.
ViT-S/16 takes 224x224 input, splits into 196 patches of 16x16 each.
Output is a 384-dim CLS token — no spatial dimensions, no GAP needed.

In [ ]:
POS_DIR    = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
NEG_DIR    = "/content/drive/MyDrive/ArchP/binary_dataset/negative"
OUTPUT_DIR = "/content/outputs"
IMG_SIZE   = 224
BATCH_SIZE = 32

VALID_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
os.makedirs(OUTPUT_DIR, exist_ok=True)

pos_files = [(str(p), 1) for p in sorted(Path(POS_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]
neg_files = [(str(p), 0) for p in sorted(Path(NEG_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]

df = pd.DataFrame(pos_files + neg_files, columns=['filepath', 'label'])

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Class weights -> Neg: {class_weight_dict[0]:.4f} | Pos: {class_weight_dict[1]:.4f}")

Train: 1898 | Val: 406 | Test: 407
Class weights -> Neg: 0.6215 | Pos: 2.5580


## Step 3 — Build tf.data Datasets
Key difference from all CNN models: ViT-S/16 from TF Hub expects
pixels normalized to [0, 1] — divide by 255.0 explicitly.
CNNs had internal normalization; ViT Hub model does not.
ignore_errors() handles the known corrupt PNG as always.

In [ ]:
def load_and_preprocess(filepath, label):
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0   # ← ViT expects [0,1]
    return image, tf.cast(label, tf.float32)

def build_dataset(dataframe, shuffle=False):
    filepaths = dataframe['filepath'].values
    labels    = dataframe['label'].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.ignore_errors()
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, shuffle=True)
val_ds   = build_dataset(val_df,   shuffle=False)
test_ds  = build_dataset(test_df,  shuffle=False)

print("Datasets built — 224x224 px, [0,1] normalized, ignore_errors active")
print(f"Train batches: {len(train_df)//BATCH_SIZE} | "
      f"Val batches: {len(val_df)//BATCH_SIZE} | "
      f"Test batches: {len(test_df)//BATCH_SIZE}")

Datasets built — 224x224 px, [0,1] normalized, ignore_errors active
Train batches: 59 | Val batches: 12 | Test batches: 12


## Step 4 — Build ViT-Small Model
Architecture is fundamentally different from all CNN models:
- TF Hub downloads ViT-S/16 pretrained on ImageNet-21k (~22M params)
- No convolutional layers — pure self-attention across 196 patches
- Output: 384-dim CLS token (already flat — no GAP needed)
- Head: Dense(256, ReLU) → Dropout(0.4) → Dense(1, Sigmoid)
- Augmentation applied BEFORE the Hub layer
- hub.KerasLayer(trainable=False) freezes the ViT in Stage 1

Note: First run downloads ~85MB weights from TF Hub — takes ~30s.
Weights are cached for subsequent runs in the same session.

In [ ]:
class ViTSmallModel(keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Augmentation layers
        self.aug = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(factor=0.25, fill_mode='reflect'),
            layers.RandomBrightness(factor=0.15),
            layers.RandomContrast(factor=0.15),
            layers.RandomZoom(height_factor=(-0.1, 0.1), fill_mode='reflect'),
        ], name="augmentation")
        # ViT-S/16 backbone
        self.vit = hub.KerasLayer(VIT_URL, trainable=False, name="vit_s16")
        # Head
        self.head_bn      = layers.BatchNormalization(name="head_bn")
        self.head_dense   = layers.Dense(256, activation='relu', name="head_dense")
        self.head_dropout = layers.Dropout(0.4, name="head_dropout")
        self.output_layer = layers.Dense(1, activation='sigmoid', name="output")

    def call(self, inputs, training=False):
        x = self.aug(inputs, training=training)
        x = self.vit(x, training=training)   # → 384-dim CLS token
        x = self.head_bn(x, training=training)
        x = self.head_dense(x)
        x = self.head_dropout(x, training=training)
        return self.output_layer(x)

    def freeze_backbone(self):
        self.vit.trainable = False

    def unfreeze_backbone(self):
        self.vit.trainable = True

print("Building model — downloading ViT-S/16 weights (~30s)...")
model = ViTSmallModel(name="ViT_Small_geoglyph")

# Build by passing a dummy batch so weights initialize
dummy = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
_ = model(dummy, training=False)

# Count parameters
total     = sum(tf.size(v).numpy() for v in model.trainable_variables +
                model.non_trainable_variables)
trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Non-trainable:    {total - trainable:,}")
print("Model built successfully.")

Building model — downloading ViT-S/16 weights (~30s)...
Total params:     100,365
Trainable params: 99,585
Non-trainable:    780
Model built successfully.


## Step 5 — Stage 1 Training (Frozen ViT Backbone)
Only the ~99K head trains. ViT backbone completely frozen.
Key difference from CNNs: no BN freeze fix needed in Stage 2
— ViT has no BatchNormalization layers in its backbone.
LR=1e-3, EarlyStopping patience=7, ReduceLROnPlateau patience=3.

In [ ]:
checkpoint_s1 = "/content/drive/MyDrive/ArchP/vit_small_stage1_best.weights.h5"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_s1, monitor='val_auc_roc', mode='max',
    save_best_only=True, save_weights_only=True, verbose=1)

earlystop_cb = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc', mode='max',
    patience=7, restore_best_weights=True, verbose=1)

reducelr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc', mode='max',
    factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb, earlystop_cb, reducelr_cb],
    verbose=1
)

Epoch 1/30
     60/Unknown 240s 4s/step - accuracy: 0.5217 - auc_roc: 0.4920 - loss: 0.9860 - precision: 0.1876 - recall: 0.4805

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_auc_roc improved from None to 0.57808, saving model to /content/drive/MyDrive/ArchP/vit_small_stage1_best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/ArchP/vit_small_stage1_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 300s 5s/step - accuracy: 0.5100 - auc_roc: 0.4885 - loss: 0.9395 - precision: 0.1905 - recall: 0.4636 - val_accuracy: 0.3556 - val_auc_roc: 0.5781 - val_loss: 1.1569 - val_precision: 0.2138 - val_recall: 0.8608 - learning_rate: 0.0010
Epoch 2/30
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - accuracy: 0.4016 - auc_roc: 0.4674 - loss: 0.8314 - precision: 0.1724 - recall: 0.5359
Epoch 2: val_auc_roc improved from 0.57808 to 0.60061, saving model to /content/drive/MyDrive/ArchP/vit_small_stage1_best.weights.h5

Epoch 2: finished saving model to /content/drive/MyDrive/ArchP/vit_small_stage1_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 39s 641ms/step - accuracy: 0.4568 - auc_roc: 0.4821 - loss: 0.8468 - precision: 0.1815 - recall: 0.5067 - val_ac

## Step 6 — Stage 2 Training (Fine-Tuning ViT Backbone)
Load best Stage 1 weights (Epoch 19), unfreeze the ViT backbone.
No BN freeze needed — ViT uses LayerNorm, not BatchNorm.
LR=1e-5 — attention weights are very sensitive, smaller LR than CNNs.
This is the critical test: can self-attention adapt to satellite imagery?

In [14]:
model.load_weights(checkpoint_s1)

# Unfreeze ViT backbone — no BN freeze needed (ViT uses LayerNorm)
model.unfreeze_backbone()

checkpoint_s2 = "/content/drive/MyDrive/ArchP/vit_small_stage2_best.weights.h5"

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

checkpoint_cb_s2 = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_s2, monitor='val_auc_roc', mode='max',
    save_best_only=True, save_weights_only=True, verbose=1)

earlystop_cb_s2 = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc', mode='max',
    patience=8, restore_best_weights=True, verbose=1)

reducelr_cb_s2 = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc', mode='max',
    factor=0.5, patience=3, min_lr=1e-7, verbose=1)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb_s2, earlystop_cb_s2, reducelr_cb_s2],
    verbose=1
)

Epoch 1/30
     60/Unknown 36s 520ms/step - accuracy: 0.5085 - auc_roc: 0.5249 - loss: 0.6793 - precision: 0.1885 - recall: 0.5093
Epoch 1: val_auc_roc improved from None to 0.67324, saving model to /content/drive/MyDrive/ArchP/vit_small_stage2_best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/ArchP/vit_small_stage2_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 44s 654ms/step - accuracy: 0.5011 - auc_roc: 0.5033 - loss: 0.7012 - precision: 0.1956 - recall: 0.4987 - val_accuracy: 0.6494 - val_auc_roc: 0.6732 - val_loss: 0.6356 - val_precision: 0.2886 - val_recall: 0.5443 - learning_rate: 1.0000e-05
Epoch 2/30
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - accuracy: 0.5011 - auc_roc: 0.4897 - loss: 0.7029 - precision: 0.1876 - recall: 0.4842
Epoch 2: val_auc_roc improved from 0.67324 to 0.67407, saving model to /content/drive/MyDrive/ArchP/vit_small_stage2_best.weights.h5

Epoch 2: finished saving model to /content/drive/MyDrive/ArchP/vit_small_stage2_best.weights.h5
60/

Why this happened — the real explanation:

ViT has no convolutional inductive bias. CNNs assume that nearby pixels are related — which is exactly true for geoglyphs (local geometric edges, corners, lines). ViT makes no such assumption and must learn spatial locality entirely from data. With only 371 positive training examples, there simply isn't enough data for 22M attention parameters to learn that local geometric patterns matter. The model never discovered what a geoglyph looks like.

This is compounded by the patch size — each 16×16 patch covers approximately 32×32 meters of ground at 2m/pixel. A geoglyph edge might span only 1-2 patches, giving the attention mechanism very little signal to work with globally.

In [15]:
model.load_weights(checkpoint_s2)

results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'accuracy', 'auc_roc', 'precision', 'recall']

print("\n===== ViT-Small Official Test Results =====")
for name, val in zip(metric_names, results):
    print(f"  {name:12s}: {val:.4f}")

precision = results[3]
recall    = results[4]
f1 = 2 * (precision * recall) / (precision + recall) \
     if (precision + recall) > 0 else 0
print(f"  {'f1_score':12s}: {f1:.4f}")
print("===========================================")

13/13 ━━━━━━━━━━━━━━━━━━━━ 58s 4s/step - accuracy: 0.6880 - auc_roc: 0.7102 - loss: 0.5981 - precision: 0.3233 - recall: 0.5375

===== ViT-Small Official Test Results =====
  loss        : 0.5981
  accuracy    : 0.6880
  auc_roc     : 0.7102
  precision   : 0.3233
  recall      : 0.5375
  f1_score    : 0.4038
